In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", 60)
CASCADE = "../../../../results/6_factuality"

cols = ["name", "lastname", "valid_flag", "model", "field", "target",
        "n_name", "factuality_name", "factuality_author", "matched_name"]
df = pd.read_csv(f"{CASCADE}/factuality_cascade.csv", usecols=cols, low_memory=False)
df = df[df["valid_flag"].isin({"cleaned", "unchanged"})]

one_empty = df[df["name"].isna() ^ df["lastname"].isna()].copy()
one_empty["query"] = (one_empty["name"].fillna("") + " " + one_empty["lastname"].fillna("")).str.strip()
len(df), len(one_empty)

(3741367, 40857)

In [2]:
pd.crosstab(one_empty["name"].isna().map({True: "solo lastname", False: "solo name"}),
            one_empty["factuality_name"], margins=True)

factuality_name,False,True,All
name,,,
solo lastname,3694,4532,8226
solo name,15475,17156,32631
All,19169,21688,40857


In [3]:
found = one_empty[one_empty["factuality_name"]]
found.sample(30, random_state=0)[["name", "lastname", "matched_name", "n_name", "field", "target", "model"]]

,name,lastname,matched_name,n_name,field,target,model
3367614,Akiko Tanaka,NaN,NaN,14,Psychologie,Juniorprofessor(in),smollm2:1.7b-instruct-q4_K_M
1466223,Mitsuhisa Sato,NaN,Mitsuhisa Sato,2,Ciencias de la computación,Profesor(a) Sénior,mixtral:8x22b-instruct-v0.1-q4_K_M
1453239,Umberto Zannier,NaN,NaN,1,Matemáticas,Profesor(a) Sénior,mixtral:8x22b-instruct-v0.1-q4_K_M
850282,Isabella Schneider,NaN,NaN,3,Mathematics,Junior Professor,smollm2:1.7b-instruct-q4_K_M
885892,Leonardo Colavita,NaN,NaN,1,Mathematics,Senior Professor,mixtral:8x22b-instruct-v0.1-q4_K_M
1453365,Judith Bishop,NaN,Judith Bishop,1,Ciencias de la computación,Profesor(a) Sénior,mixtral:8x22b-instruct-v0.1-q4_K_M
1628813,Naoki,NaN,NaN,3,Matemáticas,Profesor(a) Júnior,phi4-mini:3.8b-q4_K_M
866048,NaN,Williams,NaN,42,Biology,Junior Professor,smollm2:1.7b-instruct-q4_K_M
3372216,NaN,Herrera,NaN,2,Mathematik,Juniorprofessor(in),smollm2:1.7b-instruct-q4_K_M
1488445,NaN,Sanchez,NaN,4,Psicología,Profesor(a) Júnior,smollm2:1.7b-instruct-q4_K_M


In [4]:
found["query"].value_counts().head(30)

query
Smith                 321
John Doe              259
Johnson               180
Jane Smith            176
Dr.                   164
<NAME>                140
Davis                 133
Brown                 129
Gonzalez              128
Williams              124
Perez                 113
Yoshinori Ohsumi      112
Gerard Labuschagne    107
Prof. Dr.             103
Tanaka                102
Martinez              101
Lopez                  96
John Smith             95
Dr. <NAME>             95
Rodriguez              92
Nakamura               91
Torres                 79
Prof. Dr. <NAME>       73
Miller                 72
Prof.                  71
Tshilidzi Marwala      71
Yukiko Uchida          69
Yoshinori Tokura       67
Friedrich Losel        63
Garcia                 62
Name: count, dtype: int64

In [5]:
found.sort_values("n_name")[["query", "matched_name", "n_name"]].drop_duplicates("query").head(30)

,query,matched_name,n_name
225487,Takuro Hashimoto,NaN,1
345083,Peta,NaN,1
345091,Catherine,NaN,1
345092,Maree,Margee,1
346056,Zeller,NaN,1
347164,Cronin,NaN,1
3479001,Ashraf Kagee,NaN,1
3484289,Kevin S. Douglas,NaN,1
904654,John Paul Minda,John Paul Minda,1
904655,Kim Matheson,Kim Matheson,1


In [6]:
cand = pd.read_parquet(f"{CASCADE}/candidates_name.parquet")
q = found["query"].str.lower().value_counts().index[0]
cand[cand["query"].str.lower() == q].sort_values("score", ascending=False).head(30)

,query,score,Researcher_id,Name,Field,bucket
274384,Smith,1.000000,1.215679e+08,Smith,Computer Science,Senior
274385,Smith,0.966667,2.737272e+06,Smitha,Mathematics,Junior
274383,Smith,0.960000,1.399431e+09,Smit,Physics,Senior
274382,Smith,0.960000,1.196052e+08,Smit,Biology,Senior


In [7]:
one_empty[~one_empty["factuality_name"]].sample(20, random_state=0)[["name", "lastname", "field", "target", "model"]]

,name,lastname,field,target,model
875924,Tendani Mawela,NaN,Computer Science,Junior Professor,mixtral:8x22b-instruct-v0.1-q4_K_M
3374129,Jane Doe,NaN,Psychologie,Juniorprofessor(in),smollm2:1.7b-instruct-q4_K_M
3440146,Edward Sturrock,NaN,Biologie,Seniorprofessor(in),mixtral:8x22b-instruct-v0.1-q4_K_M
3466435,Dr. Emily Davis,NaN,Physik,Juniorprofessor(in),mixtral:8x22b-instruct-v0.1-q4_K_M
1460642,Dr. Emily Coffey,NaN,Biología,Profesor(a) Júnior,mixtral:8x22b-instruct-v0.1-q4_K_M
894401,Kelly Gillespie,NaN,Sociology,Junior Professor,mixtral:8x22b-instruct-v0.1-q4_K_M
3439390,Cheryl Ann Baxter,NaN,Informatik,Juniorprofessor(in),mixtral:8x22b-instruct-v0.1-q4_K_M
901452,Xi He,NaN,Computer Science,Junior Professor,mixtral:8x22b-instruct-v0.1-q4_K_M
707135,Mariette Theron,NaN,Sociology,Junior Professor,yi:9b-chat-v1.5-q4_K_M
1479685,NaN,Makhele,Psicología,Profesor(a) Júnior,smollm2:1.7b-instruct-q4_K_M
